In [ ]:
# 1. 安装 Colab 运行所需依赖（首个 Cell：保留 Colab 原生 PyTorch/CUDA，仅安装兼容版本依赖）
import importlib
import subprocess
import sys
import torch

subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q",
    "anndata>=0.10.7,<0.12",
    "scanpy>=1.9.3",
    "diffusers==0.17.1",
    "huggingface-hub==0.25.2",
    "torch-geometric>=2.3.1",
])

torch_ver = torch.__version__.split("+")[0]
parts = torch_ver.split(".")
torch_candidates = [torch_ver]
if len(parts) == 3 and parts[2] != "0":
    torch_candidates.append(f"{parts[0]}.{parts[1]}.0")

cuda_ver = (
    f"cu{torch.version.cuda.replace('.', '')}"
    if torch.cuda.is_available() and torch.version.cuda
    else "cpu"
)
print(f"PyTorch: {torch.__version__}, CUDA: {torch.version.cuda} ({cuda_ver})")

installed_backend = None
for tv in torch_candidates:
    pyg_wheel_url = f"https://data.pyg.org/whl/torch-{tv}+{cuda_ver}.html"
    for pkg in ("pyg-lib", "torch-sparse"):
        ret = subprocess.run([
            sys.executable, "-m", "pip", "install", "-q",
            "--only-binary=:all:", pkg, "-f", pyg_wheel_url,
        ])
        if ret.returncode == 0:
            installed_backend = pkg
            print(f"已安装 PyG 采样后端 {pkg} (wheel index: {pyg_wheel_url})")
            break
    if installed_backend is not None:
        break

if installed_backend is None:
    raise RuntimeError(
        f"未能安装预编译的 pyg-lib 或 torch-sparse（PyTorch={torch.__version__}, CUDA={cuda_ver}）。"
        "请在 Colab 中确认已开启 GPU 运行时，或手动指定匹配的 PyG wheel 索引。"
    )

import anndata  # noqa: F401
import h5py  # noqa: F401
import scanpy  # noqa: F401
import torch_geometric  # noqa: F401
from diffusers.models.unet_1d_blocks import get_down_block  # noqa: F401
importlib.import_module(installed_backend.replace("-", "_"))

print(f"依赖安装并验证完成！GPU 可用: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU 型号: {torch.cuda.get_device_name(0)}")


# STADiffuser Zhuang-ABCA-2 Colab 训练与预测

- **运行时建议**：请在 Colab 菜单选择 `代码执行程序 -> 更改运行时类型 -> GPU`（推荐 L4 / A100，T4 也可运行）。
- **数据路径**：默认读取 Google Drive 中的 `MyDrive/Dataset/Zhuang-ABCA-2`。
- **结果持久化**：预处理文件、每轮断点权重（`checkpoints/`）、验证指标日志（`*.jsonl`）以及所有测试片预测结果（`predicted_section_*.h5ad`）均直接保存到 Google Drive。若 Colab 断开重连，重新运行即可从最后一个完成的 epoch 或测试切片自动断点续训。


In [ ]:
# 2. 挂载 Google Drive 并配置数据、权重与测试结果路径
import subprocess
from pathlib import Path
from google.colab import drive

drive.mount("/content/drive")

# Google Drive 中的数据集目录（对应: 我的云端硬盘 > Dataset > Zhuang-ABCA-2）
DRIVE_DATA_DIR = Path("/content/drive/MyDrive/Dataset/Zhuang-ABCA-2")

# 所有预处理产物、模型权重、日志与测试预测结果均保存到 Google Drive
DRIVE_OUTPUT_ROOT = Path("/content/drive/MyDrive/STADiffuser_outputs/zhuang_abca2")
PREPARED_DIR = DRIVE_OUTPUT_ROOT / "prepared"
OUTPUT_DIR = DRIVE_OUTPUT_ROOT / "run"

# 建议保持 True：将原始 h5ad/csv 缓存到 Colab 本地磁盘以加速 h5py 读取并避免 Drive FUSE IO 波动；
# 注意输出目录 OUTPUT_DIR 与 PREPARED_DIR 仍在 Google Drive 上，所有结果都会持久保存。
CACHE_DATA_LOCAL = True
LOCAL_DATA_DIR = Path("/content/data/Zhuang-ABCA-2")

# 设为 True 可先做 1 epoch 冒烟测试（使用独立输出目录），正式训练请保持 False
SMOKE_TEST = False
if SMOKE_TEST:
    OUTPUT_DIR = DRIVE_OUTPUT_ROOT / "run_smoke"

FORCE_PREPARE = False

required_files = [
    DRIVE_DATA_DIR / "ccf_coordinates.csv",
    DRIVE_DATA_DIR / "cell_metadata_with_cluster_annotation.csv",
]
missing = [str(p) for p in required_files if not p.is_file()]
h5ads = sorted(DRIVE_DATA_DIR.glob("sec_*.h5ad"))
if missing or not h5ads:
    raise FileNotFoundError(
        f"请检查数据集目录 {DRIVE_DATA_DIR}：缺少 {missing}，找到 sec_*.h5ad 数量={len(h5ads)}"
    )
print(f"在 Google Drive 找到 {len(h5ads)} 个切片 h5ad 文件: {DRIVE_DATA_DIR}")

if CACHE_DATA_LOCAL:
    LOCAL_DATA_DIR.mkdir(parents=True, exist_ok=True)
    print(f"正在同步数据集到 Colab 本地磁盘 {LOCAL_DATA_DIR}（已存在的文件会自动跳过）...")
    subprocess.run(
        ["rsync", "-ah", "--update", f"{DRIVE_DATA_DIR}/", f"{LOCAL_DATA_DIR}/"],
        check=True,
    )
    DATA_DIR = LOCAL_DATA_DIR
else:
    DATA_DIR = DRIVE_DATA_DIR

PREPARED_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"DATA_DIR     = {DATA_DIR}")
print(f"PREPARED_DIR = {PREPARED_DIR}")
print(f"OUTPUT_DIR   = {OUTPUT_DIR}")


In [ ]:
# 3. 拉取 STADiffuser 仓库并以 --no-deps 安装本地包
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/HuangMaokujiji-AAA/STADiffuser.git"
# 如果你的仓库是私有仓库或已上传到 Google Drive，可将 REPO_DIR 直接改为 Drive 路径
REPO_DIR = Path("/content/STADiffuser")

if not REPO_DIR.exists():
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
else:
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=True)

os.chdir(REPO_DIR)
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "--no-deps", "-e", "."])

import experiments.zhuang_abca2.zhuang  # noqa: F401
import experiments.zhuang_abca2.stadiffuser_run  # noqa: F401
print(f"工作目录已切换至 {Path.cwd()}，模块导入正常。")


In [ ]:
# 4. 数据预处理：CCF 坐标匹配、固定切片划分与切片元数据缓存（保存到 Google Drive）
import subprocess
import sys
import pandas as pd

def run_streamed(cmd):
    proc = subprocess.Popen(
        cmd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    for line in proc.stdout:
        print(line, end="", flush=True)
    ret = proc.wait()
    if ret != 0:
        raise RuntimeError(f"命令执行失败，退出码 {ret}: {' '.join(cmd)}")

if FORCE_PREPARE or not (PREPARED_DIR / "prepared.json").is_file():
    print("开始执行 Zhuang-ABCA-2 预处理...")
    run_streamed([
        sys.executable, "-u", "-m", "experiments.zhuang_abca2.zhuang",
        str(DATA_DIR), str(PREPARED_DIR),
    ])
else:
    print(f"检测到已完成的预处理结果，跳过预处理：{PREPARED_DIR / 'prepared.json'}")

split_df = pd.read_csv(PREPARED_DIR / "split.csv")
print("\n切片划分统计:")
print(split_df.groupby("split")[["section", "cells", "raw_cells"]].agg({"section": "count", "cells": "sum", "raw_cells": "sum"}))


In [ ]:
# 5. 启动 STADiffuser 训练与测试片预测（自动断点续训，所有权重和结果直接保存到 Google Drive）
import sys
import torch

DEVICE = "cuda:0" if torch.cuda.is_available() else "cpu"
if DEVICE == "cpu" and not SMOKE_TEST:
    raise RuntimeError("当前未检测到 CUDA GPU，请在 Colab 中切换为 GPU 运行时后再启动正式训练。")

cmd = [
    sys.executable, "-u", "-m", "experiments.zhuang_abca2.stadiffuser_run",
    str(DATA_DIR),
    str(PREPARED_DIR),
    str(OUTPUT_DIR),
    "--device", DEVICE,
]
if SMOKE_TEST:
    cmd.append("--smoke")

print("执行命令:", " ".join(cmd))
run_streamed(cmd)


In [ ]:
# 6. 检查 Google Drive 中已保存的权重、日志与测试片预测结果
import json

ckpt_dir = OUTPUT_DIR / "checkpoints"
print("=== 权重文件 (Google Drive) ===")
if ckpt_dir.is_dir():
    for p in sorted(ckpt_dir.iterdir()):
        size_mb = p.stat().st_size / (1024 * 1024)
        print(f"  {p.name:35s} {size_mb:8.2f} MB")

pred_files = sorted(OUTPUT_DIR.glob("predicted_section_*.h5ad"))
print(f"\n=== 测试片预测结果 ({len(pred_files)} 个文件, Google Drive) ===")
for p in pred_files:
    size_mb = p.stat().st_size / (1024 * 1024)
    print(f"  {p.name:35s} {size_mb:8.2f} MB")

completed_path = OUTPUT_DIR / "completed.json"
if completed_path.is_file():
    print("\n=== 训练完成摘要 (completed.json) ===")
    print(json.dumps(json.loads(completed_path.read_text()), ensure_ascii=False, indent=2))
